# 02 · Fetch and extract the official-scope data, then audit it

Reads the archives through your Drive shortcut (or downloads them with `gdown` if the shortcut is
unavailable) and extracts **only** the 9 official-scope designs and the 6 routability maps the project uses,
into `/content/circuitnet_ispd15` (local disk, rebuilt each session; ≈ 15–30 min). Nothing on Drive is
modified. Archive integrity: byte sizes are checked against the Phase 0 audit, and every gzip stream is
read to its end (tar reports truncation/CRC errors). The audit then checks sample counts and map shapes.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json
from verihgn_repro.colab import access_check, fetch_archives, extract_for_scope, write_json
from verihgn_repro import reporting
SCOPE = 'official9'          # D6 primary scope (never mixed with all14)
CACHE_ON_DRIVE = False       # I-2: True only if notebook 01 showed > ~15 GB free on Drive
acc = access_check('/content/drive')
METHOD = 'drive' if acc['ok'] else 'gdown'
print('method:', METHOD, '| archive size checks:', [(a['archive'], a['ok']) for a in acc['archives']])
paths = fetch_archives(L, method=METHOD, ispd15_dir=acc['found']['path'], cache_on_drive=CACHE_ON_DRIVE)

In [ ]:
counts = extract_for_scope(paths, L, scope=SCOPE)
print(json.dumps(counts, indent=1))

## Audit: discovery (expected vs found) and label/feature map shapes for every sample

In [ ]:
import numpy as np
from verihgn_repro.data import circuitnet as cn
from verihgn_repro.utils.safe_npy import load_npy
raw = cn.RawLayout(L.local_raw)
disc = reporting.discovery_report(L.local_raw, SCOPE)
problems = [] if disc['complete'] else [f'discovery incomplete: {disc["per_design"]}']
shapes = {}
for s in raw.samples():
    try:
        h = load_npy(raw.routability('gr_h_overflow', s)); v = load_npy(raw.routability('gr_v_overflow', s))
        if h.shape != v.shape:
            problems.append(f'{s}: H {h.shape} != V {v.shape}')
        for k in ('cell_density', 'rudy', 'rudy_pin', 'macro_map'):
            if load_npy(raw.routability(k, s)).shape != h.shape:
                problems.append(f'{s}: {k} shape differs from label')
        if not raw.placement_micron(s).exists():
            problems.append(f'{s}: micron placement missing')
        shapes.setdefault(cn.parse_sample_name(s)['design'], set()).add(h.shape)
    except Exception as e:
        problems.append(f'{s}: {type(e).__name__}: {e}')
audit = {'discovery': disc, 'extraction': counts, 'archive_checks': acc['archives'], 'method': METHOD,
         'label_shapes': {d: sorted(map(list, v)) for d, v in shapes.items()}, 'problems': problems}
write_json(audit, L.reports / f'02_audit_{SCOPE}.json')
print(json.dumps(disc, indent=1)); print('problems:', problems[:10])
assert not problems, 'audit failed — do not continue'
print(f"✔ {disc['found_total']}/{disc['expected_total']} samples discovered, all maps aligned")